# Run a scenario pack

No Pygame window. Builds every spec in a pack, runs a planner, shows a table.

Until A* / D* Lite exist, use `straight_line` or `none`. `straight_line` only succeeds when the last-seen map allows a straight shot — so many rows will be `success=False`. That is expected.

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

import pandas as pd
from swarmpathfinding.bench.harness import run_pack, write_csv
from swarmpathfinding.generators.packs import list_packs, pack_specs

print(list_packs())

## Choose pack, algorithm, seeds

`limit` caps how many specs run (useful while iterating). Drop it to run the whole pack.

In [ ]:
PACK = "slow_edit"       # single_static | slow_edit | rts_shared_goal | physics_crowd | fow_discover | fow_ghost | opposite
ALGO = "straight_line"
SEEDS = (0,)
LIMIT = 8                # None = all

preview = pack_specs(PACK, seeds=SEEDS)
print(f"{PACK}: {len(preview)} specs")
for spec in preview[:LIMIT or None]:
    print(" ", spec.scenario_id)

In [ ]:
rows = run_pack(PACK, ALGO, seeds=SEEDS, limit=LIMIT)
df = pd.DataFrame([r.as_dict() for r in rows])
df

In [ ]:
cols = [c for c in ["scenario_id", "map_kind", "density", "n_agents", "success", "path_length", "plan_ms", "replan_ms"] if c in df.columns]
df[cols]

In [ ]:
out = Path("benches") / "out" / f"{PACK}.csv"
write_csv(out, rows)
print("wrote", out.resolve())

## Open one row in the live lab

Blocks until you close Pygame.

In [ ]:
from swarmpathfinding.viz.live import launch_spec

INDEX = 0
launch_spec(preview[INDEX], planner=ALGO)